In [ ]:
# 1. Load Dataset

!pip install -q openpyxl tensorflow scikit-learn

import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split

# 2. Download and Load Dataset

url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

!wget -q "$url" -O online_retail.zip
!unzip -o -q online_retail.zip

df = pd.read_excel("Online Retail.xlsx")

print("Dataset loaded successfully!")
print("Original dataset shape:", df.shape)

print("\nFirst 5 records:")
print(df.head())

# 3. Select Required Columns

df = df[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
]

# 4. Data Cleaning

df = df.dropna(subset=["CustomerID"])

df = df[
    ~df["InvoiceNo"].astype(str).str.startswith("C")
]

df = df[df["Quantity"] > 0]

df = df[df["UnitPrice"] > 0]

df = df.dropna(subset=["Description"])

print("\nAfter cleaning:")
print("Dataset shape:", df.shape)

# 5. Create Purchase Amount

df["TotalAmount"] = (
    df["Quantity"] * df["UnitPrice"]
)

print(
    "\nTotal Sales:",
    round(df["TotalAmount"].sum(), 2)
)

# 6. Create Customer-Product Interactions

interaction_df = df.groupby(
    ["CustomerID", "StockCode"]
).agg(
    Quantity=("Quantity", "sum"),
    TotalAmount=("TotalAmount", "sum")
).reset_index()

print("\nCustomer-Product interactions:")
print(interaction_df.head())

# 7. Encode Customer IDs

customers = interaction_df["CustomerID"].unique()
products = interaction_df["StockCode"].unique()

customer_to_index = {
    customer: index
    for index, customer in enumerate(customers)
}

product_to_index = {
    product: index
    for index, product in enumerate(products)
}

index_to_customer = {
    index: customer
    for customer, index in customer_to_index.items()
}

index_to_product = {
    index: product
    for product, index in product_to_index.items()
}

interaction_df["CustomerIndex"] = (
    interaction_df["CustomerID"].map(
        customer_to_index
    )
)

interaction_df["ProductIndex"] = (
    interaction_df["StockCode"].map(
        product_to_index
    )
)

# 8. Create Positive Interactions

positive_df = interaction_df[
    [
        "CustomerIndex",
        "ProductIndex"
    ]
].copy()

positive_df["Purchase"] = 1

positive_pairs = set(
    zip(
        positive_df["CustomerIndex"],
        positive_df["ProductIndex"]
    )
)

# 9. Generate Negative Samples

np.random.seed(42)

num_customers = len(customers)
num_products = len(products)

negative_samples = []

target_negative_samples = len(
    positive_df
)

while len(negative_samples) < target_negative_samples:

    customer_index = np.random.randint(
        0,
        num_customers
    )

    product_index = np.random.randint(
        0,
        num_products
    )

    if (
        customer_index,
        product_index
    ) not in positive_pairs:

        negative_samples.append(
            [
                customer_index,
                product_index,
                0
            ]
        )

negative_df = pd.DataFrame(
    negative_samples,
    columns=[
        "CustomerIndex",
        "ProductIndex",
        "Purchase"
    ]
)

# 10. Combine Positive and Negative Data

final_df = pd.concat(
    [
        positive_df,
        negative_df
    ],
    ignore_index=True
)

final_df = final_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nFinal interaction dataset:")
print(final_df.head())

print(
    "\nTotal interaction records:",
    len(final_df)
)

# 11. Prepare Training Data

X_customer = final_df[
    "CustomerIndex"
].values

X_product = final_df[
    "ProductIndex"
].values

y = final_df[
    "Purchase"
].values

# 12. Train-Test Split

(
    X_customer_train,
    X_customer_test,
    X_product_train,
    X_product_test,
    y_train,
    y_test
) = train_test_split(
    X_customer,
    X_product,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 13. Define Model Parameters

num_customers = len(customers)
num_products = len(products)

embedding_dim = 32

print(
    "\nNumber of customers:",
    num_customers
)

print(
    "Number of products:",
    num_products
)

# 14. Customer Embedding

customer_input = Input(
    shape=(1,),
    name="customer_input"
)

customer_embedding = Embedding(
    input_dim=num_customers,
    output_dim=embedding_dim,
    name="customer_embedding"
)(customer_input)

customer_vector = Flatten()(
    customer_embedding
)

# 15. Product Embedding

product_input = Input(
    shape=(1,),
    name="product_input"
)

product_embedding = Embedding(
    input_dim=num_products,
    output_dim=embedding_dim,
    name="product_embedding"
)(product_input)

product_vector = Flatten()(
    product_embedding
)

# 16. Combine Embeddings

combined = Concatenate()(
    [
        customer_vector,
        product_vector
    ]
)

# 17. Deep Neural Network

x = Dense(
    128,
    activation="relu"
)(combined)

x = Dropout(0.3)(x)

x = Dense(
    64,
    activation="relu"
)(x)

x = Dropout(0.2)(x)

x = Dense(
    32,
    activation="relu"
)(x)

output = Dense(
    1,
    activation="sigmoid"
)(x)

# 18. Create Model

model = Model(
    inputs=[
        customer_input,
        product_input
    ],
    outputs=output
)

# 19. Compile Model

model.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nDeep Learning Recommendation Model:")
model.summary()

# 20. Train Model

print("\nTraining Recommendation Model...")

history = model.fit(
    [
        X_customer_train,
        X_product_train
    ],
    y_train,
    epochs=10,
    batch_size=512,
    validation_split=0.2,
    verbose=1
)

# 21. Evaluate Model

test_loss, test_accuracy = model.evaluate(
    [
        X_customer_test,
        X_product_test
    ],
    y_test,
    verbose=0
)

print(
    "\nTest Loss:",
    round(test_loss, 4)
)

print(
    "Test Accuracy:",
    round(test_accuracy * 100, 2),
    "%"
)

# 22. Recommendation Function

def recommend_products(
    customer_id,
    top_n=10
):

    if customer_id not in customer_to_index:

        print(
            "Customer ID not found."
        )

        return

    customer_index = customer_to_index[
        customer_id
    ]

    purchased_products = set(
        interaction_df[
            interaction_df[
                "CustomerID"
            ] == customer_id
        ]["StockCode"]
    )

    candidate_products = []

    for product_index in range(
        num_products
    ):

        product_id = index_to_product[
            product_index
        ]

        if product_id not in purchased_products:

            candidate_products.append(
                product_index
            )

    customer_array = np.array(
        [customer_index] *
        len(candidate_products)
    )

    product_array = np.array(
        candidate_products
    )

    predictions = model.predict(
        [
            customer_array,
            product_array
        ],
        verbose=0
    ).flatten()

    recommendation_df = pd.DataFrame({
        "StockCode": [
            index_to_product[i]
            for i in candidate_products
        ],
        "Score": predictions
    })

    recommendation_df = (
        recommendation_df
        .sort_values(
            "Score",
            ascending=False
        )
        .head(top_n)
    )

    description_map = (
        df.groupby("StockCode")
        ["Description"]
        .first()
    )

    recommendation_df[
        "Description"
    ] = recommendation_df[
        "StockCode"
    ].map(description_map)

    print(
        "\nRecommended Products for Customer:",
        customer_id
    )

    print(
        recommendation_df[
            [
                "StockCode",
                "Description",
                "Score"
            ]
        ].to_string(
            index=False
        )
    )

    return recommendation_df

# 23. Generate Recommendations

example_customer = (
    interaction_df[
        "CustomerID"
    ].iloc[0]
)

recommend_products(
    example_customer,
    top_n=10
)

Dataset loaded successfully!
Original dataset shape: (541909, 8)

First 5 records:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  

After cleaning:
Dataset shape: (397884, 8)

Total Sales: 8911407.9

Customer-Product interactions:
   Cus

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ customer_input      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ customer_embedding  │ (None, 1, 32)     │    138,816 │ customer_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_embedding   │ (None, 1, 32)     │    117,280 │ product_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 32)        │          0 │ customer_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 32)        │          0 │ product_embeddin… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 64)        │          0 │ flatten_2[0][0],  │
│ (Concatenate)       │                   │            │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │      8,320 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 32)        │      2,080 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1)         │         33 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 274,785 (1.05 MB)

 Trainable params: 274,785 (1.05 MB)

 Non-trainable params: 0 (0.00 B)


Training Recommendation Model...
Epoch 1/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.7608 - loss: 0.4868 - val_accuracy: 0.7898 - val_loss: 0.4447
Epoch 2/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.7957 - loss: 0.4342 - val_accuracy: 0.7913 - val_loss: 0.4391
Epoch 3/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.8010 - loss: 0.4226 - val_accuracy: 0.7951 - val_loss: 0.4367
Epoch 4/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.8079 - loss: 0.4115 - val_accuracy: 0.7951 - val_loss: 0.4359
Epoch 5/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.8177 - loss: 0.3971 - val_accuracy: 0.7955 - val_loss: 0.4391
Epoch 6/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.8298 - loss: 0.3787 - val_accuracy: 0.7939 - val_loss: 0.4457
Epoch 7/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.8423 - loss: 0.3583 - val_accuracy: 0.7935 - val_loss: 0.4563
Epoch 8/10
667/667 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accurac

,StockCode,Score,Description
895,85123A,0.859284,WHITE HANGING HEART T-LIGHT HOLDER
1212,22457,0.753844,NATURAL SLATE HEART CHALKBOARD
1063,48138,0.750678,DOORMAT UNION FLAG
910,22111,0.723996,SCOTTIE DOG HOT WATER BOTTLE
411,23356,0.713723,LOVE HOT WATER BOTTLE
1223,85066,0.669005,CREAM SWEETHEART MINI CHEST
214,22178,0.643878,VICTORIAN GLASS HANGING T-LIGHT
1158,84970L,0.626732,SINGLE HEART ZINC T-LIGHT HOLDER
886,84879,0.593459,ASSORTED COLOUR BIRD ORNAMENT
1319,84755,0.567987,COLOUR GLASS T-LIGHT HOLDER HANGING
